# SUSI score by district-year

SUSI = 환경점수 + 안전점수 + 소음점수 + (교통점수/4)

In [45]:
import pandas as pd
from pathlib import Path

base = Path('.')

env_df = pd.read_csv(base / '환경점수_가중치.csv')
safety_df = pd.read_csv(base / 'gu_safety_burden_score_25_by_year_clean.csv')
noise_df = pd.read_csv(base / 'noise_vibration_final_long_no_rank.csv')
traffic_df = pd.read_csv(base / '교통_혼잡_스트레스_종합점수_통합_2022_2024.csv')

print('loaded rows:', len(env_df), len(safety_df), len(noise_df), len(traffic_df))

loaded rows: 900 75 75 75


In [46]:
def clean_cols(df):
    d = df.copy()
    d.columns = d.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)
    return d

def clean_gu(x):
    s = str(x).strip()
    s = s.replace('\u3000', ' ').replace(' ', '')
    s = s.replace('서울특별시', '')
    return s

def parse_year(x):
    y = pd.to_numeric(pd.Series([x]).astype(str).str.extract(r'(\d{4})')[0], errors='coerce').iloc[0]
    return int(y) if pd.notna(y) else pd.NA

def to_score(df, gu_col, year_col, score_col, out_col):
    t = df[[gu_col, year_col, score_col]].copy()
    t[gu_col] = t[gu_col].map(clean_gu)
    t[year_col] = t[year_col].map(parse_year).astype('Int64')
    t = t.dropna(subset=[year_col]).copy()
    t[year_col] = t[year_col].astype(int)
    t[out_col] = pd.to_numeric(t[score_col], errors='coerce')
    t = t.rename(columns={gu_col:'자치구', year_col:'연도'})
    t = t[['자치구', '연도', out_col]].dropna()
    t = t.groupby(['자치구', '연도'], as_index=False)[out_col].mean()
    return t.sort_values(['자치구', '연도']).reset_index(drop=True)

In [47]:
env_df = clean_cols(env_df)
safety_df = clean_cols(safety_df)
noise_df = clean_cols(noise_df)
traffic_df = clean_cols(traffic_df)

env_score = to_score(env_df, '자치구', '연도', '환경점수', '환경점수')
safety_score = to_score(safety_df, '자치구', '연도', '안전부담점수_25점', '안전점수')

if '지표' in noise_df.columns:
    noise_df = noise_df[noise_df['지표'].astype(str).str.contains('총점', na=False)].copy()
noise_score = to_score(noise_df, '자치구', '연도', '점수', '소음점수')

traffic_score = to_score(traffic_df, '자치구', '연도', '교통_혼잡스트레스점수', '교통점수')

print('rows by domain:', len(env_score), len(safety_score), len(noise_score), len(traffic_score))

rows by domain: 75 75 75 75


In [48]:
result = env_score.merge(safety_score, on=['자치구', '연도'], how='inner')
result = result.merge(noise_score, on=['자치구', '연도'], how='inner')
result = result.merge(traffic_score, on=['자치구', '연도'], how='inner')

result['교통점수_25점'] = result['교통점수'] / 4
result['스시점수'] = result['환경점수'] + result['안전점수'] + result['소음점수'] + result['교통점수_25점']

num_cols = ['환경점수', '안전점수', '소음점수', '교통점수', '교통점수_25점', '스시점수']
result[num_cols] = result[num_cols].round(2)
result = result.sort_values(['자치구', '연도']).reset_index(drop=True)

print('result rows:', len(result))
result

result rows: 75


,자치구,연도,환경점수,안전점수,소음점수,교통점수,교통점수_25점,스시점수
0,강남구,2022,6.34,12.20,19.31,62.74,15.68,53.52
1,강남구,2023,4.13,11.92,20.90,63.31,15.83,52.78
2,강남구,2024,5.67,11.73,19.36,57.95,14.49,51.25
3,강동구,2022,10.70,6.70,11.88,35.20,8.80,38.08
4,강동구,2023,6.46,7.03,10.96,36.49,9.12,33.57
5,강동구,2024,8.46,7.17,10.73,37.35,9.34,35.70
6,강북구,2022,10.96,8.06,11.65,38.04,9.51,40.17
7,강북구,2023,8.76,8.00,9.49,38.89,9.72,35.97
8,강북구,2024,11.04,7.87,10.48,40.10,10.02,39.42
9,강서구,2022,6.78,6.58,14.05,37.45,9.36,36.78


In [49]:
susi_2022 = result[result['연도'] == 2022].copy().reset_index(drop=True)
susi_2023 = result[result['연도'] == 2023].copy().reset_index(drop=True)
susi_2024 = result[result['연도'] == 2024].copy().reset_index(drop=True)

print(len(susi_2022), len(susi_2023), len(susi_2024))
display(susi_2022)
display(susi_2023)
display(susi_2024)

25 25 25


,자치구,연도,환경점수,안전점수,소음점수,교통점수,교통점수_25점,스시점수
0,강남구,2022,6.34,12.20,19.31,62.74,15.68,53.52
1,강동구,2022,10.70,6.70,11.88,35.20,8.80,38.08
2,강북구,2022,10.96,8.06,11.65,38.04,9.51,40.17
3,강서구,2022,6.78,6.58,14.05,37.45,9.36,36.78
4,관악구,2022,10.20,8.92,10.83,42.00,10.50,40.45
5,광진구,2022,10.66,9.13,11.30,41.76,10.44,41.52
6,구로구,2022,8.73,8.16,10.58,33.22,8.30,35.77
7,금천구,2022,10.23,9.89,8.69,14.72,3.68,32.50
8,노원구,2022,9.11,7.01,8.12,36.14,9.04,33.28
9,도봉구,2022,10.52,6.38,10.08,28.20,7.05,34.04


,자치구,연도,환경점수,안전점수,소음점수,교통점수,교통점수_25점,스시점수
0,강남구,2023,4.13,11.92,20.90,63.31,15.83,52.78
1,강동구,2023,6.46,7.03,10.96,36.49,9.12,33.57
2,강북구,2023,8.76,8.00,9.49,38.89,9.72,35.97
3,강서구,2023,4.31,6.62,13.89,46.34,11.58,36.41
4,관악구,2023,6.94,9.08,9.76,43.76,10.94,36.72
5,광진구,2023,6.99,9.42,9.95,40.76,10.19,36.55
6,구로구,2023,6.93,9.08,10.85,35.25,8.81,35.67
7,금천구,2023,7.32,9.14,9.57,14.90,3.72,29.76
8,노원구,2023,6.10,7.16,8.96,35.78,8.94,31.16
9,도봉구,2023,7.85,7.09,9.48,28.24,7.06,31.48


,자치구,연도,환경점수,안전점수,소음점수,교통점수,교통점수_25점,스시점수
0,강남구,2024,5.67,11.73,19.36,57.95,14.49,51.25
1,강동구,2024,8.46,7.17,10.73,37.35,9.34,35.70
2,강북구,2024,11.04,7.87,10.48,40.10,10.02,39.42
3,강서구,2024,5.65,6.65,12.72,44.88,11.22,36.24
4,관악구,2024,8.63,8.94,10.03,41.89,10.47,38.07
5,광진구,2024,7.98,8.21,9.57,39.46,9.86,35.62
6,구로구,2024,8.17,9.09,9.67,33.44,8.36,35.30
7,금천구,2024,8.09,8.64,7.64,13.87,3.47,27.83
8,노원구,2024,6.93,6.85,8.84,36.13,9.03,31.65
9,도봉구,2024,10.07,6.75,9.42,28.20,7.05,33.29


In [50]:
env_score.rename(columns={'환경점수': '점수'}).to_csv(base / 'env_score_gu_year.csv', index=False, encoding='utf-8-sig')
safety_score.rename(columns={'안전점수': '점수'}).to_csv(base / 'safety_score_gu_year.csv', index=False, encoding='utf-8-sig')
noise_score.rename(columns={'소음점수': '점수'}).to_csv(base / 'noise_score_gu_year.csv', index=False, encoding='utf-8-sig')
traffic_score.rename(columns={'교통점수': '점수'}).to_csv(base / 'traffic_score_gu_year.csv', index=False, encoding='utf-8-sig')
result.to_csv(base / 'gu_susi_score_100_by_year.csv', index=False, encoding='utf-8-sig')
print('saved all outputs')

saved all outputs


In [51]:
susi_2022.sort_values("스시점수", ascending=False)


,자치구,연도,환경점수,안전점수,소음점수,교통점수,교통점수_25점,스시점수
23,중구,2022,8.25,24.19,16.03,56.56,14.14,62.61
0,강남구,2022,6.34,12.20,19.31,62.74,15.68,53.52
22,종로구,2022,10.61,16.07,13.95,51.42,12.86,53.48
20,용산구,2022,9.25,13.01,13.84,55.09,13.77,49.88
12,마포구,2022,9.46,11.16,14.28,56.70,14.18,49.08
19,영등포구,2022,8.28,12.77,13.92,54.27,13.57,48.53
14,서초구,2022,8.31,10.13,15.37,55.94,13.98,47.80
10,동대문구,2022,9.79,9.29,15.51,30.27,7.57,42.15
5,광진구,2022,10.66,9.13,11.30,41.76,10.44,41.52
15,성동구,2022,8.57,7.24,14.53,41.14,10.28,40.62


In [52]:
susi_2023.sort_values("스시점수", ascending=False)


,자치구,연도,환경점수,안전점수,소음점수,교통점수,교통점수_25점,스시점수
23,중구,2023,5.45,24.14,11.14,58.40,14.60,55.33
0,강남구,2023,4.13,11.92,20.90,63.31,15.83,52.78
20,용산구,2023,7.62,12.93,14.78,59.90,14.98,50.30
22,종로구,2023,7.27,14.92,15.28,50.41,12.60,50.07
19,영등포구,2023,6.75,11.02,14.94,60.36,15.09,47.80
14,서초구,2023,5.74,11.09,15.21,56.15,14.04,46.08
12,마포구,2023,6.97,11.00,12.89,56.75,14.19,45.04
15,성동구,2023,6.78,7.98,15.48,42.94,10.74,40.97
10,동대문구,2023,7.68,8.02,16.42,32.06,8.02,40.14
13,서대문구,2023,9.32,8.22,12.52,36.00,9.00,39.07


In [53]:
susi_2024.sort_values("스시점수", ascending=False)

,자치구,연도,환경점수,안전점수,소음점수,교통점수,교통점수_25점,스시점수
23,중구,2024,6.93,24.56,11.47,56.12,14.03,56.98
20,용산구,2024,9.10,15.59,13.75,59.21,14.80,53.24
22,종로구,2024,8.59,15.22,15.20,49.83,12.46,51.46
0,강남구,2024,5.67,11.73,19.36,57.95,14.49,51.25
12,마포구,2024,7.66,11.65,14.57,59.10,14.78,48.65
19,영등포구,2024,8.23,10.98,13.70,53.98,13.50,46.41
14,서초구,2024,7.64,9.52,15.96,51.13,12.78,45.90
15,성동구,2024,8.91,8.90,16.43,41.75,10.44,44.69
10,동대문구,2024,9.04,9.47,16.77,32.44,8.11,43.39
13,서대문구,2024,9.21,7.95,14.22,35.83,8.96,40.34
